# 03 — Chains with LCEL
LangChain Expression Language (LCEL) lets you compose Runnables using the `|` pipe operator.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


## 1. Simple Chain: Prompt → Model → Parser

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarise the text in exactly one sentence."),
        ("human",  "{text}")
    ])
    | llm
    | parser
)

result = chain.invoke({"text": "LangChain is a framework for building LLM-powered applications using composable, modular components such as prompts, chains, memory, agents, and tools."})
print(result)


## 2. Sequential Chain (output of step 1 → input of step 2)

In [ ]:
# Step 1: generate a blog title
title_chain = (
    ChatPromptTemplate.from_template("Generate one compelling blog title about: {topic}")
    | llm | parser
)

# Step 2: write an intro for that title
intro_chain = (
    ChatPromptTemplate.from_template("Write a 2-sentence intro for the blog titled: {title}")
    | llm | parser
)

# Compose: output of title_chain feeds into intro_chain
full_chain = title_chain | (lambda title: {"title": title}) | intro_chain

print(full_chain.invoke({"topic": "quantum computing for beginners"}))


## 3. Parallel Chain with RunnableParallel

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Run two chains in parallel on the same input
parallel_chain = RunnableParallel({
    "pros": (
        ChatPromptTemplate.from_template("List 3 pros of {technology} in bullet points.")
        | llm | parser
    ),
    "cons": (
        ChatPromptTemplate.from_template("List 3 cons of {technology} in bullet points.")
        | llm | parser
    ),
})

result = parallel_chain.invoke({"technology": "microservices architecture"})
print("PROS:\n", result["pros"])
print("\nCONS:\n", result["cons"])


## 4. Chain with Pydantic Output Parser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List

# Define the expected output schema
class CodeReview(BaseModel):
    overall_quality: str = Field(description="poor / fair / good / excellent")
    issues: List[str]    = Field(description="list of issues found")
    suggestions: List[str] = Field(description="list of improvement suggestions")

parser = JsonOutputParser(pydantic_object=CodeReview)

review_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a senior Python code reviewer. {format_instructions}"),
        ("human",  "Review this code:\n```python\n{code}\n```")
    ]).partial(format_instructions=parser.get_format_instructions())
    | llm
    | parser
)

sample_code = """
def divide(a, b):
    return a / b
"""

review = review_chain.invoke({"code": sample_code})
print("Quality  :", review["overall_quality"])
print("Issues   :", review["issues"])
print("Suggestions:", review["suggestions"])


---
**Key takeaway:** LCEL chains are lazy — they don't execute until `.invoke()` is called. This enables streaming, batching, and async out of the box.